# Final Experiment: Repeated Split Stability (Center + No Intercept + Shared Grid)

This notebook runs repeated random target/source splits and reports stability statistics.

Settings kept fixed for fairness:
- `y` mode = center
- no intercept
- ridge `tau` and lasso `alpha` are selected from the same shared grid (mini-style)
- two directions are both evaluated:
  1. Direction A: target = `X`, source = `X1`
  2. Direction B: target = `X1`, source = `X`


In [ ]:
import numpy as np
import pandas as pd
import warnings

from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score
from sklearn.exceptions import ConvergenceWarning

import os, sys
_EXP_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _EXP_ROOT not in sys.path:
    sys.path.insert(0, _EXP_ROOT)
from transrr_lib.robust_ridge_optimizer import solve_robust_ridge
from transrr_lib.find_tau_opt import find_optimal_tau_robust_ridge

warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [ ]:
# Fixed config
RANDOM_STATE = 10
STEP = 4
NUM_SAMPLE_ROWS = 160
WHITEN_EPS = 1e-6

# Repeated split stability
N_REPEATS = 20
REPEAT_SEED_START = 1000

# Robust loss shape
DELTA_PARAM = 1.35
ETA_PARAM = 0.1

# Fair shared interval (same style as mini)
COMMON_GRID = np.logspace(-4, 1, 11)

# Final choices
Y_MODE = "center"
USE_INTERCEPT = False

print("COMMON_GRID:", COMMON_GRID)
print(f"N_REPEATS={N_REPEATS}, REPEAT_SEED_START={REPEAT_SEED_START}")


In [ ]:
# Load raw data
X_df = pd.read_csv("shootout/X.csv")
Xt_df = pd.read_csv("shootout/test_X.csv")
X1_df = pd.read_csv("shootout/X_1.csv")
X1t_df = pd.read_csv("shootout/test_X1.csv")

y_df = pd.read_csv("shootout/y.csv")
yt_df = pd.read_csv("shootout/test_y.csv")

y_train_full = y_df.iloc[:, 2].to_numpy()
y_test_full = yt_df.iloc[:, 2].to_numpy()

X_target_train_raw = X_df.to_numpy()
X_target_test_raw = Xt_df.to_numpy()
X_source_train_raw = X1_df.to_numpy()
X_source_test_raw = X1t_df.to_numpy()

# Merge target/source train+test first
X_target_all = np.vstack([X_target_train_raw, X_target_test_raw])
X_source_all = np.vstack([X_source_train_raw, X_source_test_raw])

# Variable selection: every 4th predictor
selected_idx = np.arange(0, X_target_all.shape[1], STEP)
X_target_sel_all = X_target_all[:, selected_idx]
X_source_sel_all = X_source_all[:, selected_idx]

print("Merged shapes:")
print("target_all:", X_target_all.shape, "source_all:", X_source_all.shape)
print("Selected predictor count:", len(selected_idx))


In [ ]:
# Split back + domain-wise whitening (fit on each train only)
def fit_whitener(X, eps=1e-6):
    mean = X.mean(axis=0)
    X_centered = X - mean
    cov = np.cov(X_centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals, eps, None)
    W = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    return mean, W

n_target_train = X_target_train_raw.shape[0]
n_target_test = X_target_test_raw.shape[0]
n_source_train = X_source_train_raw.shape[0]

X_target_train_sel = X_target_sel_all[:n_target_train]
X_target_test_sel = X_target_sel_all[n_target_train:n_target_train + n_target_test]
X_source_train_sel = X_source_sel_all[:n_source_train]
X_source_test_sel = X_source_sel_all[n_source_train:]

t_mean, t_W = fit_whitener(X_target_train_sel, eps=WHITEN_EPS)
s_mean, s_W = fit_whitener(X_source_train_sel, eps=WHITEN_EPS)

X_target_train = (X_target_train_sel - t_mean) @ t_W
X_target_test = (X_target_test_sel - t_mean) @ t_W
X_source_train = (X_source_train_sel - s_mean) @ s_W
X_source_test = (X_source_test_sel - s_mean) @ s_W

print("Whitened domain shapes:")
print("target train/test:", X_target_train.shape, X_target_test.shape)
print("source train/test:", X_source_train.shape, X_source_test.shape)


In [ ]:
# y center helpers
def fit_y_center(y):
    return {"mu": float(np.mean(y))}


def center_y(y, stats):
    return y - stats["mu"]


def uncenter_y(y_t, stats):
    return y_t + stats["mu"]


In [ ]:
# Model helpers: center + no intercept

def fit_robust_center_no_intercept(X, y, tau_grid, delta, eta):
    y_stats = fit_y_center(y)
    y_t = center_y(y, y_stats)

    tau_opt = find_optimal_tau_robust_ridge(
        X, y_t, tau_range=tau_grid, psi_delta=delta, psi_eta=eta
    )
    initial_guess = np.linalg.solve(
        X.T @ X / len(y_t) + tau_opt * np.eye(X.shape[1]),
        X.T @ y_t / len(y_t)
    )
    beta = solve_robust_ridge(X, y_t, tau_opt, delta, eta, initial_beta=initial_guess)

    return {"beta": beta, "tau": float(tau_opt), "y_stats": y_stats}


def predict_robust_center_no_intercept(model, X):
    pred_t = X @ model["beta"]
    return uncenter_y(pred_t, model["y_stats"])


def fit_lasso_center_no_intercept(X, y, alpha_grid, random_state):
    y_stats = fit_y_center(y)
    y_t = center_y(y, y_stats)

    lasso = LassoCV(
        fit_intercept=False,
        max_iter=50000,
        tol=1e-4,
        alphas=alpha_grid,
        cv=5,
        selection="cyclic",
        random_state=random_state,
    ).fit(X, y_t)

    return {"model": lasso, "alpha": float(lasso.alpha_), "y_stats": y_stats}


def predict_lasso_center_no_intercept(model, X):
    pred_t = model["model"].predict(X)
    return uncenter_y(pred_t, model["y_stats"])


In [ ]:
# One split, one direction

def run_one_split(direction_name, target_train, target_test, source_train, split_seed):
    n_train = min(target_train.shape[0], source_train.shape[0], len(y_train_full))
    n_test = min(target_test.shape[0], len(y_test_full))

    target_train_use = target_train[:n_train]
    source_train_use = source_train[:n_train]
    y_train_use = y_train_full[:n_train]

    target_test_use = target_test[:n_test]
    y_test_use = y_test_full[:n_test]

    rng = np.random.default_rng(split_seed)
    all_indices = np.arange(n_train)
    div_indices = rng.choice(all_indices, size=NUM_SAMPLE_ROWS, replace=False)
    non_div_indices = np.setdiff1d(all_indices, div_indices)

    train_X = target_train_use[div_indices]
    train_y = y_train_use[div_indices]
    train_X1 = source_train_use[non_div_indices]
    train_y1 = y_train_use[non_div_indices]
    test_X = target_test_use
    test_y = y_test_use

    # 1) Single Robust Ridge
    model_sr = fit_robust_center_no_intercept(train_X, train_y, COMMON_GRID, DELTA_PARAM, ETA_PARAM)
    pred_sr = predict_robust_center_no_intercept(model_sr, test_X)

    # 2) Transfer Robust Ridge
    model_w = fit_robust_center_no_intercept(train_X1, train_y1, COMMON_GRID, DELTA_PARAM, ETA_PARAM)
    pred_source_on_target = predict_robust_center_no_intercept(model_w, train_X)
    Y_adjusted = train_y - pred_source_on_target
    model_diff = fit_robust_center_no_intercept(train_X, Y_adjusted, COMMON_GRID, DELTA_PARAM, ETA_PARAM)
    pred_tr = predict_robust_center_no_intercept(model_w, test_X) + predict_robust_center_no_intercept(model_diff, test_X)

    # 3) Pooled Robust Ridge
    XX = np.vstack((train_X, train_X1))
    YY = np.concatenate((train_y, train_y1))
    model_pr = fit_robust_center_no_intercept(XX, YY, COMMON_GRID, DELTA_PARAM, ETA_PARAM)
    pred_pr = predict_robust_center_no_intercept(model_pr, test_X)

    # 4) Single Lasso
    lasso_sl = fit_lasso_center_no_intercept(train_X, train_y, COMMON_GRID, split_seed)
    pred_sl = predict_lasso_center_no_intercept(lasso_sl, test_X)

    # 5) Transfer Lasso
    lasso_t1 = fit_lasso_center_no_intercept(train_X1, train_y1, COMMON_GRID, split_seed)
    pred_l1_on_target = predict_lasso_center_no_intercept(lasso_t1, train_X)
    Y_offset_adjusted = train_y - pred_l1_on_target
    lasso_t2 = fit_lasso_center_no_intercept(train_X, Y_offset_adjusted, COMMON_GRID, split_seed)
    pred_tl = predict_lasso_center_no_intercept(lasso_t1, test_X) + predict_lasso_center_no_intercept(lasso_t2, test_X)

    preds = {
        "single_ridge": pred_sr,
        "transfer_ridge": pred_tr,
        "pooled_ridge": pred_pr,
        "single_lasso": pred_sl,
        "transfer_lasso": pred_tl,
    }

    rows = []
    for model_name, pred in preds.items():
        mse = float(np.mean((test_y - pred) ** 2))
        rmse = float(np.sqrt(mse))
        r2 = float(r2_score(test_y, pred))
        rows.append({
            "direction": direction_name,
            "repeat_id": int(split_seed),
            "model": model_name,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2,
            "tau_sr": model_sr["tau"],
            "tau_tr_stage1": model_w["tau"],
            "tau_tr_stage2": model_diff["tau"],
            "tau_pr": model_pr["tau"],
            "alpha_sl": lasso_sl["alpha"],
            "alpha_tl_stage1": lasso_t1["alpha"],
            "alpha_tl_stage2": lasso_t2["alpha"],
        })

    return rows


def run_repeated_direction(direction_name, target_train, target_test, source_train, n_repeats, seed_start):
    rows = []
    for r in range(n_repeats):
        split_seed = seed_start + r
        rows.extend(run_one_split(direction_name, target_train, target_test, source_train, split_seed))
    return rows


In [ ]:
# Run repeated experiments for both directions
rows_A = run_repeated_direction(
    direction_name="A_target_is_X",
    target_train=X_target_train,
    target_test=X_target_test,
    source_train=X_source_train,
    n_repeats=N_REPEATS,
    seed_start=REPEAT_SEED_START,
)

rows_B = run_repeated_direction(
    direction_name="B_target_is_X1",
    target_train=X_source_train,
    target_test=X_source_test,
    source_train=X_target_train,
    n_repeats=N_REPEATS,
    seed_start=REPEAT_SEED_START,
)

all_repeat_df = pd.DataFrame(rows_A + rows_B)
all_repeat_df.head()


In [ ]:
# Summary: mean and std RMSE (plus MSE/R2 summary)
summary_df = (
    all_repeat_df
    .groupby(["direction", "model"], as_index=False)
    .agg(
        mean_RMSE=("RMSE", "mean"),
        std_RMSE=("RMSE", "std"),
        mean_MSE=("MSE", "mean"),
        std_MSE=("MSE", "std"),
        mean_R2=("R2", "mean"),
        std_R2=("R2", "std"),
    )
    .sort_values(["direction", "mean_RMSE"])
)
summary_df


In [ ]:
# Fairness and report
print("Final setting:")
print(f"  Y_MODE={Y_MODE}, USE_INTERCEPT={USE_INTERCEPT}")
print(f"  Shared grid range=[{COMMON_GRID.min():.1e}, {COMMON_GRID.max():.1e}], n={len(COMMON_GRID)}")
print(f"  Repeated splits per direction: {N_REPEATS}")
print()

print("Average RMSE +/- SD by direction:")
for direction in summary_df["direction"].unique():
    print(f"\n[{direction}]")
    sub = summary_df[summary_df["direction"] == direction]
    for _, row in sub.iterrows():
        print(f"{row['model']:16s} | RMSE={row['mean_RMSE']:.4f} +/- {row['std_RMSE']:.4f} | R2={row['mean_R2']:.4f} +/- {row['std_R2']:.4f}")


In [ ]:
# Optional: parameter-selection frequencies from shared grid
param_cols = ["tau_sr", "tau_tr_stage1", "tau_tr_stage2", "tau_pr", "alpha_sl", "alpha_tl_stage1", "alpha_tl_stage2"]

for direction in all_repeat_df["direction"].unique():
    print(f"\n[{direction}] selected-parameter frequencies")
    sub = all_repeat_df[all_repeat_df["direction"] == direction]
    # keep one row per repeat for parameter columns (same values repeated over 5 models within a split)
    one_per_repeat = sub.drop_duplicates(subset=["repeat_id"])[["repeat_id"] + param_cols]
    for c in param_cols:
        vc = one_per_repeat[c].value_counts().sort_index()
        vc_str = ", ".join([f"{k:.6g}:{int(v)}" for k, v in vc.items()])
        print(f"  {c:15s} -> {vc_str}")


In [ ]:
# Save raw per-split results so downstream analysis (boxplots, Adaptive selector, etc.) does not require re-running the 20 splits.
import os, json
os.makedirs("res", exist_ok=True)

# 1) full per-split / per-model dataframe (RMSE/MSE/R2 + selected tau/alpha)
all_repeat_df.to_csv("res/realdata_per_split.csv", index=False)

# 2) summary table (mean/std) shown above
summary_df.to_csv("res/realdata_summary.csv", index=False)

# 3) compact JSON: n_splits x n_methods RMSE matrix per direction (for boxplot / adaptive layer)
method_names = list(all_repeat_df["model"].unique())
rmse_data = {
    "method_names": method_names,
    "n_splits": int(N_REPEATS),
    "preprocessing": "every-4th-wavelength + per-domain whitening",
    "directions": {},
}
for direction in all_repeat_df["direction"].unique():
    sub = all_repeat_df[all_repeat_df["direction"] == direction]
    rmse_matrix = (
        sub.pivot(index="repeat_id", columns="model", values="RMSE")
        .reindex(columns=method_names)
        .to_numpy()
        .tolist()
    )
    rmse_data["directions"][direction] = rmse_matrix

with open("res/realdata_rmse.json", "w") as f:
    json.dump(rmse_data, f, indent=2)

print("Saved:")
print("  res/realdata_per_split.csv")
print("  res/realdata_summary.csv")
print("  res/realdata_rmse.json")
